# Part 3 - Model Analysis

This notebook covers steady states, bifurcation analysis, and structural network analysis.


# Part 3.1 - Steady State Computations

Catalyst computes polynomial-system steady states through homotopy continuation.

We load HomotopyContinuation with `import` (rather than `using`) to avoid name conflicts with `solve`.


In [ ]:
using Catalyst
import HomotopyContinuation

wilhelm_2009_model = @reaction_network begin
    k1, Y --> 2X
    k2, 2X --> X + Y
    k3, X + Y --> Y
    k4, X --> 0
end

ps = [:k1 => 8.0, :k2 => 2.0, :k3 => 1.0, :k4 => 1.5]
steady_states = hc_steady_states(wilhelm_2009_model, ps)
steady_states


# Part 3.2 - Bifurcation Analysis

Catalyst interfaces with BifurcationKit through `BifurcationProblem`.

For robust continuation, we first compute a steady-state guess by long-time ODE simulation.


In [ ]:
using OrdinaryDiffEq, BifurcationKit, Plots

bistable_switch = @reaction_network begin
    v0 + hill(X, v, K, n), 0 --> X
    deg, X --> 0
end

p_start = [:v0 => 0.1, :v => 2.5, :K => 75.0, :n => 2.0, :deg => 0.01]
sol_guess = solve(ODEProblem(bistable_switch, [:X => 1.0], (0.0, 4000.0), p_start), Rodas5P())
u_guess = [:X => sol_guess[:X][end]]

bif_prob = BifurcationProblem(bistable_switch, u_guess, p_start, :K; plot_var = :X)
opts_br = ContinuationPar(max_steps = 5000, p_min = 10.0, p_max = 200.0)
bf = bifurcationdiagram(bif_prob, PALC(), 2, opts_br; bothside = true)

plot(bf; xguide = "K", yguide = "X", title = "Bifurcation diagram")


# Part 3.3 - Network Analysis

Catalyst provides structural analysis utilities, including conservation laws and deficiency.


In [ ]:
using Latexify

two_state_model = @reaction_network begin
    (k1, k2), X1 <--> X2
end

latexify(two_state_model; form = :ode)
conservedequations(two_state_model)

ps = [:k1 => 2.0, :k2 => 1.0]
u0 = [:X1 => 1.0, :X2 => 1.0]
hc_steady_states(two_state_model, ps; u0)


In [ ]:
network_example = @reaction_network begin
    k1, A + B --> C
    k2, C --> A + B
    k3, C --> D
end

reactioncomplexes(network_example)
linkageclasses(network_example)
deficiency(network_example)
